<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_classification/18_5_MutliClassClassification/18_5_9_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multiclass Classification: Exercise
## Wine Quality Prediction

---

## Overview

In this exercise you will apply the complete multiclass classification workflow to a real dataset: **Red Wine Quality** from the UCI Machine Learning Repository.

The dataset contains chemical measurements of 1,599 red wines. Each wine was rated by at least three wine experts and assigned a quality score from 3 to 8. Your job is to build a classifier that predicts quality level from chemistry alone.

**Your tasks:**
1. Explore the data and check the class distribution
2. Create a three-class target (Low / Medium / High)
3. Prepare features and split the data
4. Train a baseline XGBoost model and evaluate it
5. Handle class imbalance with sample weights
6. Compare models with macro F1
7. Answer reflection questions

Work through the cells in order. Each task has starter code and a hidden solution you can reveal if you get stuck.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (ConfusionMatrixDisplay, classification_report,
                              f1_score, confusion_matrix)
from sklearn.utils import compute_sample_weight

sns.set_theme(style="whitegrid")

---

## Task 1: Load and Explore the Data

The wine quality data is hosted at the UCI Machine Learning Repository. The separator is a semicolon (`;`), not a comma.

Load the data, inspect the first few rows, and answer: how many rows and features does this dataset have?

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"

# TODO: Load the data using pd.read_csv — remember the sep=";" argument
# wine = ...

# TODO: Print the shape and show the first few rows
# print(f"Shape: {wine.shape}")
# wine.head()

In [ ]:
# @title Execute to see solution
solution = '''
wine = pd.read_csv(url, sep=";")
print(f"Shape: {wine.shape}")
wine.head()
'''
print(solution)

Once you have loaded the data, look at the `quality` column. What values does it take?

In [ ]:
# TODO: Show the distribution of quality scores
# Hint: wine["quality"].value_counts().sort_index()

In [ ]:
# @title Execute to see solution
solution = '''
print(wine["quality"].value_counts().sort_index())

# Bar chart
counts = wine["quality"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(counts.index, counts.values, color="#4C72B0", edgecolor="white")
ax.set_xlabel("Quality Score")
ax.set_ylabel("Number of Wines")
ax.set_title("Distribution of Wine Quality Scores")
for score, count in zip(counts.index, counts.values):
    ax.text(score, count + 5, str(count), ha="center", fontweight="bold")
plt.tight_layout()
plt.show()
'''
print(solution)

---

## Task 2: Create a Three-Class Target

Raw quality scores from 3–8 give you up to six classes, most of which have very few examples. That is too many classes for a meaningful model.

Bin the scores into three groups:
- **Low**: quality ≤ 5
- **Medium**: quality = 6
- **High**: quality ≥ 7

After binning, compute the naive baseline accuracy (always predict the most common class).

In [ ]:
# TODO: Create a new column "quality_group" with values "Low", "Medium", "High"
# Hint: use pd.cut() or np.where()

# wine["quality_group"] = ...

# TODO: Print the class distribution and compute the naive baseline
# class_names = ["Low", "Medium", "High"]

In [ ]:
# @title Execute to see solution
solution = '''
def assign_group(q):
    if q <= 5:
        return "Low"
    elif q == 6:
        return "Medium"
    else:
        return "High"

wine["quality_group"] = wine["quality"].apply(assign_group)

class_names = ["Low", "Medium", "High"]
counts = wine["quality_group"].value_counts()
print(counts)
print()

naive_baseline = counts.max() / len(wine)
print(f"Naive baseline (always predict {counts.idxmax()}): {naive_baseline:.1%}")
'''
print(solution)

---

## Task 3: Visualize Class Separation

Pick two chemical features and make a scatterplot colored by quality group. You are looking for whether the groups form distinct clusters.

Suggestions: `alcohol` vs. `volatile acidity`, or `alcohol` vs. `sulphates`.

Does the feature space look separable?

In [ ]:
# TODO: Create a scatterplot of two features, colored by quality_group
# Hint: loop over class_names and plot each subset with a different color

colors = ["#4C72B0", "#DD8452", "#55A868"]

# fig, ax = plt.subplots(figsize=(8, 5))
# for name, color in zip(class_names, colors):
#     subset = wine[wine["quality_group"] == name]
#     ax.scatter(...)

In [ ]:
# @title Execute to see solution
solution = '''
colors = ["#4C72B0", "#DD8452", "#55A868"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for name, color in zip(class_names, colors):
    subset = wine[wine["quality_group"] == name]
    axes[0].scatter(subset["alcohol"], subset["volatile acidity"],
                   label=name, color=color, alpha=0.5, edgecolors="white", s=40)
    axes[1].scatter(subset["alcohol"], subset["sulphates"],
                   label=name, color=color, alpha=0.5, edgecolors="white", s=40)

for ax, (xlabel, ylabel, title) in zip(axes, [
    ("Alcohol (%)", "Volatile Acidity", "Alcohol vs. Volatile Acidity"),
    ("Alcohol (%)", "Sulphates", "Alcohol vs. Sulphates"),
]):
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()

plt.suptitle("Can Chemistry Separate Wine Quality Groups?", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
'''
print(solution)

---

## Task 4: Prepare Data and Train a Baseline Model

1. Create the feature matrix `X` (all columns except `quality` and `quality_group`)
2. Create the target vector `y` as integers (0=Low, 1=Medium, 2=High)
3. Split into train/test with stratification
4. Train an XGBoost multiclass classifier
5. Print the classification report

In [ ]:
# TODO: Create X and y

# label_map = {name: i for i, name in enumerate(class_names)}
# y = wine["quality_group"].map(label_map).to_numpy()
# X = wine.drop(columns=["quality", "quality_group"]).to_numpy(dtype=float)

# TODO: Train-test split (test_size=0.2, random_state=42, stratify=y)

# TODO: Train XGBoost (objective="multi:softprob", num_class=3, n_estimators=200,
#                       max_depth=5, learning_rate=0.1, random_state=42, eval_metric="mlogloss")

# TODO: Print the classification report

In [ ]:
# @title Execute to see solution
solution = '''
label_map = {name: i for i, name in enumerate(class_names)}
y = wine["quality_group"].map(label_map).to_numpy()
X = wine.drop(columns=["quality", "quality_group"]).to_numpy(dtype=float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model_plain = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric="mlogloss"
)
model_plain.fit(X_train, y_train)
y_pred_plain = model_plain.predict(X_test)

print("Baseline XGBoost:")
print(classification_report(y_test, y_pred_plain, target_names=class_names, digits=3))
'''
print(solution)

---

## Task 5: Handle Class Imbalance

1. Compute sample weights using `compute_sample_weight("balanced", y=y_train)`
2. Retrain XGBoost with those weights
3. Print the new classification report
4. Compare macro F1 before and after

In [ ]:
# TODO: Compute balanced sample weights

# TODO: Retrain with sample_weight=...

# TODO: Print classification report

# TODO: Print macro F1 for both models
# print(f"Plain macro F1   : {f1_score(y_test, y_pred_plain,    average='macro'):.3f}")
# print(f"Balanced macro F1: {f1_score(y_test, y_pred_balanced, average='macro'):.3f}")

In [ ]:
# @title Execute to see solution
solution = '''
sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

model_balanced = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric="mlogloss"
)
model_balanced.fit(X_train, y_train, sample_weight=sample_weights)
y_pred_balanced = model_balanced.predict(X_test)

print("Balanced XGBoost:")
print(classification_report(y_test, y_pred_balanced, target_names=class_names, digits=3))
print()
print(f"Plain macro F1   : {f1_score(y_test, y_pred_plain,    average='macro'):.3f}")
print(f"Balanced macro F1: {f1_score(y_test, y_pred_balanced, average='macro'):.3f}")
'''
print(solution)

---

## Task 6: Visualize the Confusion Matrix

Plot the confusion matrix for the balanced model as a heatmap. Which two classes does the model most frequently confuse?

In [ ]:
# TODO: Plot the confusion matrix for the balanced model
# Hint: Use ConfusionMatrixDisplay.from_predictions(...)

In [ ]:
# @title Execute to see solution
solution = '''
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_balanced,
    display_labels=class_names,
    cmap="Blues",
    ax=ax
)
ax.set_title("Wine Quality: Balanced XGBoost Confusion Matrix")
ax.grid(False)
plt.tight_layout()
plt.show()
'''
print(solution)

---

## Task 7: Honest Evaluation with Stratified Cross-Validation

A single train-test split can be misleading when classes are imbalanced. The "High" quality group is small, so one lucky or unlucky split could move its macro F1 several points.

Use **5-fold stratified cross-validation** to get a more reliable estimate:

1. Use `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` to preserve class proportions in each fold
2. In each fold, fit the **balanced** XGBoost (with `compute_sample_weight`) on the training portion
3. Evaluate macro F1 on the held-out fold
4. Report the mean and standard deviation across all five folds

Compare the CV macro F1 to your single-split result from Task 5. Are they close? A large gap suggests the single split was unrepresentative.

In [ ]:
# TODO: Run 5-fold stratified CV on the balanced XGBoost model
# Hint: loop over outer_cv.split(X, y), fit with sample weights, score with f1_score(..., average="macro")

# outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# cv_scores = []
#
# for train_idx, test_idx in outer_cv.split(X, y):
#     X_tr, X_te = X[train_idx], X[test_idx]
#     y_tr, y_te = y[train_idx], y[test_idx]
#     w_tr = compute_sample_weight(class_weight="balanced", y=y_tr)
#     ...
#     cv_scores.append(...)
#
# cv_scores = np.array(cv_scores)
# print(f"CV macro F1: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

In [ ]:
# @title Execute to see solution
solution = '''
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for train_idx, test_idx in outer_cv.split(X, y):
    X_tr, X_te = X[train_idx], X[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]

    w_tr = compute_sample_weight(class_weight="balanced", y=y_tr)

    m = xgb.XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        random_state=42,
        eval_metric="mlogloss"
    )
    m.fit(X_tr, y_tr, sample_weight=w_tr)
    cv_scores.append(f1_score(y_te, m.predict(X_te), average="macro"))

cv_scores = np.array(cv_scores)
print(f"5-fold CV macro F1 (balanced XGBoost): {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print()
print("The ± value (std across folds) tells you how stable the estimate is.")
print("Compare this mean to your single-split macro F1 from Task 5.")
'''
print(solution)

---

## Task 8: Reflection Questions

Answer these questions in a new markdown cell (double-click to edit), or discuss them with your group.

**Q1.** The wine quality dataset has three classes: Low, Medium, and High. After binning, which class is most common? Does this create an imbalance problem, or is the distribution close enough to balanced that it probably does not matter?

**Q2.** Compare the macro F1 and weighted F1 for your baseline model. Is the gap large or small? What does that tell you about the model's performance across classes?

**Q3.** You are helping a sommelier build a tool that flags *High* quality wines for special reserve. Of precision and recall for the "High" class, which matters more in this application? Why?

**Q4.** The XGBoost model takes raw chemical measurements and predicts quality. Do you think these predictions would generalize to wines from a different country or grape variety? What would you need to check?

**Q5.** In Unit 17, you used wine to illustrate regression (predicting exact quality scores). Here you used it for classification (predicting Low/Medium/High). Which framing makes more sense for the sommelier tool in Q3? Would a regression output (e.g., "this wine scores 7.2") be more or less useful than a three-class label?

---

## Putting It All Together

You applied the full multiclass toolkit:

| Step | What you did |
|---|---|
| Explore | Visualized class distribution, found imbalance |
| Baseline | Computed naive accuracy floor |
| Train | XGBoost with `objective="multi:softprob"` |
| Evaluate | Classification report: per-class precision/recall/F1 |
| Imbalance | Sample weights with `compute_sample_weight("balanced")` |
| Compare | Macro F1 before and after balancing |
| Confusion matrix | Identified which class pairs the model confuses |
| Cross-validation | Stratified K-Fold to verify the macro F1 estimate is stable |

The same workflow applies to any multiclass dataset you encounter. The key decisions are always the same:

1. Check the class distribution — compute the naive baseline
2. Choose macro F1 as your headline metric if classes are imbalanced or equally important
3. Use sample weights if one class is severely underrepresented
4. Read the confusion matrix to understand *which* errors the model makes, not just how many
5. Verify with stratified cross-validation — a single split may be misleading when minority classes are small